# A minimal ReAct agent

In AgentScope 2.x, `Agent` is the unified ReAct agent. This notebook creates one agent with no tools; the next lesson adds an action.

In [ ]:
import os

from dotenv import load_dotenv

from agentscope.agent import Agent, ReActConfig
from agentscope.credential import OpenAICredential
from agentscope.message import Msg, TextBlock
from agentscope.model import OpenAIChatModel

## What happens in this lesson

![Minimal AgentScope ReAct workflow](figures/minimal-react-workflow.svg)

The agent receives one user message, uses its prompt, model, and ReAct configuration to produce one assistant message. No tools are registered yet.

In [ ]:
load_dotenv()

model_name = os.getenv("MODEL")
base_url = os.getenv("OLLAMA_BASE_URL")
if not model_name or not base_url:
    raise RuntimeError("Set MODEL and OLLAMA_BASE_URL in .env before running this notebook.")

model = OpenAIChatModel(
    credential=OpenAICredential(api_key="ollama", base_url=base_url),
    model=model_name,
    stream=False,
    parameters=OpenAIChatModel.Parameters(temperature=0, max_tokens=220),
)

In [ ]:
triage_agent = Agent(
    name="triage_assistant",
    system_prompt=(
        "You are a careful digital-forensics triage assistant. "
        "Separate observed evidence from inferences. "
        "When evidence is insufficient, say so plainly."
    ),
    model=model,
    react_config=ReActConfig(max_iters=3),
)

print(f"Created: {triage_agent.name}")

In [ ]:
case_note = """
At 09:14 UTC, a workstation initiated 43 outbound connections to 198.51.100.23.
The packet capture does not include payloads or a process name.
"""

response = await triage_agent.reply(
    Msg(
        name="analyst",
        role="user",
        content=[
            TextBlock(
                text=(
                    "Review this case note. Return two short sections: "
                    "Observed evidence and Cautious inferences.\n\n"
                    f"{case_note.strip()}"
                ),
            ),
        ],
    ),
)

print("".join(block.text for block in response.content if isinstance(block, TextBlock)))